<a href="https://colab.research.google.com/github/zufarrizal/TFDrive/blob/master/TFDrive_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 TFDrive All-In-One Toolkit (Google Colab)
Unduh file besar, batch link, torrent, magnet, dan video langsung ke Google Drive pribadi Anda dengan kecepatan server Google tanpa memakan kuota internet PC.

--- 
### 🔑 Langkah 1: Hubungkan ke Google Drive
Jalankan cell di bawah ini dan klik **Connect to Google Drive** (izinkan akses akun Google Anda).

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Folder utama penyimpanan di Google Drive
BASE_DIR = '/content/drive/MyDrive/TFDrive_Downloads'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"✅ Google Drive terhubung! Folder penyimpanan: {BASE_DIR}")

--- 
### 📥 Fitur 1: Direct & Batch Downloader
Bisa memasukkan 1 link atau banyak link (pisahkan dengan Enter). Mendukung mode **Single** (aman untuk FuckingFast/MediaFire) atau **Multi-16x** (super cepat).

In [ ]:
#@title 📥 Form Direct / Batch Downloader
URLS = "" #@param {type:"string"}
MODE = "Single (Aman - wget)" #@param ["Single (Aman - wget)", "Multi-Connection 16x (aria2c)"]
SUBFOLDER = "" #@param {type:"string"}

import os

# Otomatis inisialisasi jika Langkah 1 terlewat
if 'BASE_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/TFDrive_Downloads'

dest_path = os.path.join(BASE_DIR, SUBFOLDER.strip()) if SUBFOLDER.strip() else BASE_DIR
os.makedirs(dest_path, exist_ok=True)

link_list = [u.strip() for u in URLS.strip().split('\n') if u.strip()]

if not link_list:
    print("❌ Masukkan minimal 1 URL unduhan!")
else:
    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    if "Multi" in MODE:
        !apt-get install -y -qq aria2
    
    print(f"📦 Memproses {len(link_list)} link ke folder: {dest_path}\n")
    for idx, link in enumerate(link_list, start=1):
        print(f"[{idx}/{len(link_list)}] Mengunduh: {link}")
        if "Single" in MODE:
            !wget --user-agent="{user_agent}" --referer="{link}" --content-disposition -c -P "{dest_path}" "{link}"
        else:
            headers = f'--header="User-Agent: {user_agent}"'
            !aria2c -x 16 -s 16 -k 1M -c --dir="{dest_path}" {headers} "{link}"
        print("-" * 50)
        
    print("\n🎉 Semua unduhan batch selesai! File sudah berada di Google Drive.")

--- 
### 🧲 Fitur 2: Torrent & Magnet Link Downloader
Unduh torrent dan magnet super cepat langsung ke Google Drive menggunakan `aria2c`.

In [ ]:
#@title 🧲 Form Torrent / Magnet Downloader
MAGNET_LINK = "" #@param {type:"string"}

!apt-get install -y -qq aria2

import os
from google.colab import files

# Otomatis inisialisasi jika Langkah 1 terlewat
if 'BASE_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/TFDrive_Downloads'

dest_torrent_dir = os.path.join(BASE_DIR, 'Torrents')
os.makedirs(dest_torrent_dir, exist_ok=True)

if MAGNET_LINK.strip():
    print(f"🚀 Mengunduh Magnet Link ke: {dest_torrent_dir}...")
    !aria2c --seed-time=0 --max-connection-per-server=16 --dir="{dest_torrent_dir}" "{MAGNET_LINK.strip()}"
else:
    print("📁 Silakan upload file .torrent Anda:")
    uploaded = files.upload()
    for fname in uploaded.keys():
        print(f"🚀 Mengunduh Torrent: {fname} ke: {dest_torrent_dir}...")
        !aria2c --seed-time=0 --max-connection-per-server=16 --dir="{dest_torrent_dir}" "{fname}"
        if os.path.exists(fname):
            os.remove(fname)
        break

print("\n🎉 Unduhan Torrent Selesai! Tersimpan di folder 'TFDrive_Downloads/Torrents'")

--- 
### 🗜️ Fitur 3: Ekstraktor Otomatis (RAR, ZIP, 7z, Tar.gz, Multi-Part)
Ekstrak arsip langsung di dalam Google Drive tanpa perlu unduh ke PC. Sangat cocok untuk file game multipart (`.part1.rar`).

In [ ]:
#@title 🗜️ Form Ekstraktor Arsip
NAMA_FILE_ARSIP = "" #@param {type:"string"}
FOLDER_HASIL_EKSTRAK = "Extracted" #@param {type:"string"}
PASSWORD = "" #@param {type:"string"}
HAPUS_ARSIP_SETELAH_EKSTRAK = False #@param {type:"boolean"}

!apt-get install -y -qq p7zip-full unrar

import os

# Otomatis inisialisasi jika Langkah 1 terlewat
if 'BASE_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/TFDrive_Downloads'

if not NAMA_FILE_ARSIP.strip():
    print("❌ Masukkan nama file arsip yang ada di folder TFDrive_Downloads!")
else:
    archive_file = os.path.join(BASE_DIR, NAMA_FILE_ARSIP.strip())
    dest_extract = os.path.join(BASE_DIR, FOLDER_HASIL_EKSTRAK.strip())
    os.makedirs(dest_extract, exist_ok=True)
    
    if not os.path.exists(archive_file):
        print(f"❌ File tidak ditemukan: {archive_file}")
    else:
        print(f"🗜️ Mengekstrak {NAMA_FILE_ARSIP} ke {dest_extract}...")
        pass_opt = f'-p"{PASSWORD.strip()}"' if PASSWORD.strip() else '-p-'
        
        !7z x "{archive_file}" -o"{dest_extract}" {pass_opt} -y
        
        if HAPUS_ARSIP_SETELAH_EKSTRAK:
            os.remove(archive_file)
            print(f"🗑️ Arsip asli {NAMA_FILE_ARSIP} berhasil dihapus untuk menghemat ruang.")
            
        print("\n🎉 Ekstraksi selesai! Hasil ekstrak tersimpan di Google Drive.")

--- 
### 📹 Fitur 4: Video & Audio Downloader (YouTube, TikTok, Twitter/X, IG, dll.)
Unduh video kualitas terbaik (hingga 4K) atau ekstrak MP3 langsung ke Google Drive menggunakan `yt-dlp`.

In [ ]:
#@title 📹 Form Video & Audio Downloader
VIDEO_URL = "" #@param {type:"string"}
FORMAT = "Video Kualitas Terbaik (MP4/MKV)" #@param ["Video Kualitas Terbaik (MP4/MKV)", "Audio Saja (MP3)"]

!python3 -m pip install -q yt-dlp

import os

# Otomatis inisialisasi jika Langkah 1 terlewat
if 'BASE_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/TFDrive_Downloads'

video_folder = os.path.join(BASE_DIR, 'Media_Downloads')
os.makedirs(video_folder, exist_ok=True)

if not VIDEO_URL.strip():
    print("❌ Masukkan URL video terlebih dahulu!")
else:
    output_tmpl = os.path.join(video_folder, '%(title)s.%(ext)s')
    print(f"🎬 Mengunduh video ke: {video_folder}...")
    
    if FORMAT == "Audio Saja (MP3)":
        !yt-dlp -x --audio-format mp3 -o "{output_tmpl}" "{VIDEO_URL.strip()}"
    else:
        !yt-dlp -f "bestvideo+bestaudio/best" --merge-output-format mp4 -o "{output_tmpl}" "{VIDEO_URL.strip()}"
        
    print("\n🎉 Unduhan video selesai! Tersimpan di folder 'TFDrive_Downloads/Media_Downloads'.")